# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
#Pravilo: stranica zaslužuje CTR/engagement review ako:
#(1) njena vidljivost je nedavno opala u odnosu na svoj sopstveni 90-dnevni prosek; 
#(2) njen CTR je znatno ispod proseka za njenu poziciju u pretrazi. 
#Prvo je signal opadanja, drugo je signal "vide je ali ne klikću".

#Signal 1 (moja hipoteza): stranice kod kojih impresije poslednjih 30 dana čini manje od 30% od ukupnih 90-dnevnih impresija su verovatnije da opadaju u trendu.

#Signal 2 (flag-linked, CTR-fix logika): CTR treba porediti unutar iste position_tier grupe — stranice na boljoj poziciji imaju viši CTR, pa se CTR mora gledati relativno na poziciju, ne globalno.

import pandas as pd
import numpy as np
from pathlib import Path

repo_root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "data" / "raw" / "content_refresh_anonymized.csv").exists()
)
df = pd.read_csv(repo_root / "data" / "raw" / "content_refresh_anonymized.csv")

# is_declining label — isti kao u starter pipeline-u, nikad se ne koristi kao feature
df["is_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Signal 1: recent_share = udeo poslednjih 30 dana u ukupnim 90-dnevnim impresijama
df["recent_share"] = df["impressions_last_30d"] / df["impressions_90d"]

df["recent_share_bucket"] = np.where(df["recent_share"] < 0.30, "under_30pct", "30pct_or_more")

signal1_table = df.groupby("recent_share_bucket").agg(
    declining_rate=("is_declining", "mean"),
    n=("is_declining", "size"),
).round(3)

print("Signal 1: recent_share vs is_declining")
print(signal1_table)

print("VERDICT: CONFIRMED")
print("Stranice sa recent_share < 30% imaju znatno veću stopu opadanja od ostalih.")
print("Napomena o poštenju: ovaj signal je mehanički blizak samoj definiciji trenda (obe formule zavise od impressions_last_30d), pa je deo korelacije očekivan po konstrukciji, ne čisto novo otkriće.")

# Signal 2: CTR treba porediti unutar iste pozicione grupe (CTR-fix logika sa predavanja)
# Filtriramo na stranice sa dovoljno impresija da CTR ne bude šum
visible = df[df["impressions_90d"] >= 100]

signal2_table = visible.groupby("position_tier").agg(
    mean_ctr=("ctr", "mean"),
    n=("ctr", "size"),
).sort_values("mean_ctr", ascending=False).round(4)

print("Signal 2: mean CTR by position_tier (impressions_90d >= 100)")
print(signal2_table)
print("VERDICT: CONFIRMED")
print("CTR jasno opada kako se pozicija pogoršava (top_3/page_1 imaju znatno viši CTR "
      "od page_3_5/deep) — potvrđuje logiku iza CTR-fix flega: CTR se mora porediti "
      "unutar iste position_tier grupe, nikad globalno.")

Signal 1: recent_share vs is_declining
                     declining_rate      n
recent_share_bucket                       
30pct_or_more                 0.081   9494
under_30pct                   0.755  20506
VERDICT: CONFIRMED
Stranice sa recent_share < 30% imaju znatno veću stopu opadanja od ostalih.
Napomena o poštenju: ovaj signal je mehanički blizak samoj definiciji trenda (obe formule zavise od impressions_last_30d), pa je deo korelacije očekivan po konstrukciji, ne čisto novo otkriće.
Signal 2: mean CTR by position_tier (impressions_90d >= 100)
               mean_ctr     n
position_tier                
page_1           0.3548  8633
top_3            0.3341   533
striking         0.2558  5903
page_3_5         0.1424  6058
deep             0.0554   879
VERDICT: CONFIRMED
CTR jasno opada kako se pozicija pogoršava (top_3/page_1 imaju znatno viši CTR od page_3_5/deep) — potvrđuje logiku iza CTR-fix flega: CTR se mora porediti unutar iste position_tier grupe, nikad globalno.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# Score: kombinuje oba signala u jedan transparentan broj
# Deo A: koliko je nedavna vidljivost pala (veće = gore, veći prioritet za review)
decline_component = (1 - df["recent_share"].clip(lower=0, upper=1))

# Deo B: koliko je CTR ispod proseka SVOJE position_tier grupe (position-adjusted gap)
tier_avg_ctr = df.groupby("position_tier")["ctr"].transform("mean")
ctr_gap = (tier_avg_ctr - df["ctr"]).clip(lower=0)  # samo pozitivan gap nas zanima (CTR ispod proseka)

# Normalizacija da oba dela budu uporediva (0-1 opseg)
def normalize(series):
    s = series.replace([np.inf, -np.inf], np.nan).fillna(0)
    if s.max() == s.min():
        return s * 0
    return (s - s.min()) / (s.max() - s.min())

df["decline_score_norm"] = normalize(decline_component)
df["ctr_gap_score_norm"] = normalize(ctr_gap)

# Filter: samo stranice sa dovoljno impresija da bude smisleno (izbegava šum)
df["visible_enough"] = (df["impressions_90d"] >= 100).astype(int)

df["baseline_action_score"] = (
    (0.5 * df["decline_score_norm"] + 0.5 * df["ctr_gap_score_norm"])
    * df["visible_enough"]
)

# JEDAN reason code (obavezno samo jedan po zadatku)
def reason_code(row):
    if row["visible_enough"] == 0:
        return "low_volume_skip"
    if row["decline_score_norm"] >= row["ctr_gap_score_norm"]:
        return "declining_visibility"
    return "ctr_below_position_average"

df["reason_code"] = df.apply(reason_code, axis=1)

# Action label
def action_label(reason):
    if reason == "declining_visibility":
        return "review_content_refresh"
    if reason == "ctr_below_position_average":
        return "review_ctr_metadata"
    return "monitor"

df["action"] = df["reason_code"].apply(action_label)

queue = df.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

output_columns = [
    "rank", "content_id", "client_id", "baseline_action_score", "reason_code", "action",
    "impressions_90d", "recent_share", "ctr", "position_tier", "avg_position",
    "is_declining", "trend_direction",
]

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue[output_columns].to_csv(output_path, index=False)

print(f"Wrote {len(queue):,} rows to {output_path}")
queue[output_columns].head(10)

Wrote 30,000 rows to work\outputs\baseline_action_score.csv


,rank,content_id,client_id,baseline_action_score,reason_code,action,impressions_90d,recent_share,ctr,position_tier,avg_position,is_declining,trend_direction
0,1,content_6dd1153d206b,client_7f2253d7e2,1.000000,declining_visibility,review_content_refresh,1001,0.000000,0.0,top_3,2.3,0,flat
1,2,content_baaedabd1d2f,client_7f2253d7e2,1.000000,declining_visibility,review_content_refresh,220,0.000000,0.0,top_3,0.4,0,flat
2,3,content_7288a4d4c198,client_7f2253d7e2,1.000000,declining_visibility,review_content_refresh,174,0.000000,0.0,top_3,0.1,0,flat
3,4,content_243715096698,client_f369cb89fc,1.000000,declining_visibility,review_content_refresh,143,0.000000,0.0,top_3,1.8,0,flat
4,5,content_1fddbbbaf30e,client_3fdba35f04,1.000000,declining_visibility,review_content_refresh,394,0.000000,0.0,top_3,1.7,1,down
5,6,content_6430c384931f,client_f369cb89fc,0.998634,ctr_below_position_average,review_ctr_metadata,366,0.002732,0.0,top_3,2.9,1,down
6,7,content_4e658a59c333,client_3fdba35f04,0.997487,ctr_below_position_average,review_ctr_metadata,199,0.005025,0.0,top_3,2.1,1,down
7,8,content_0d9c0ed65840,client_3fdba35f04,0.997382,ctr_below_position_average,review_ctr_metadata,382,0.005236,0.0,top_3,0.8,1,down
8,9,content_d9e4b523c0ce,client_3fdba35f04,0.996552,ctr_below_position_average,review_ctr_metadata,580,0.006897,0.0,top_3,2.9,1,down
9,10,content_93730172c7a4,client_3fdba35f04,0.996377,ctr_below_position_average,review_ctr_metadata,276,0.007246,0.0,top_3,1.4,1,down


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
top10 = queue.head(10)

# Prebrojimo pojavljivanja klijenata unutar top 10 da uhvatimo koncentraciju
client_counts_top10 = top10["client_id"].value_counts()

for row in top10.itertuples():
    print(f"#{row.rank} | {row.content_id} | action={row.action} | reason={row.reason_code} | "
          f"score={row.baseline_action_score:.3f}")
    print(f"   impressions_90d={row.impressions_90d}, recent_share={row.recent_share:.2f}, "
          f"ctr={row.ctr}, position_tier={row.position_tier}, trend_direction={row.trend_direction}")
    print(f"   Zašto je ovde: {'opala vidljivost u odnosu na sopstveni 90d prosek' if row.reason_code=='declining_visibility' else 'CTR ispod proseka za svoju poziciju'}.")

    # 1) Reason code kaže "opada", ali trend_direction ne slaže se sa tim (flat/up umesto down)
    if row.reason_code == "declining_visibility" and row.trend_direction != "down":
        print(f"   Šta bi ga učinilo pogrešnim: Reason code kaže 'opadajuća vidljivost', ali stvarni "
              f"trend_direction je '{row.trend_direction}' — ova stranica verovatno nije opala postepeno, "
              f"već je izgubila SVU vidljivost naglo negde pre više od 60 dana (možda deindeksirana, "
              f"redirect, ili tehnički problem), a ne postepeni pad zbog kvaliteta sadržaja. Akcija "
              f"'refresh sadržaja' bi ovde verovatno bila pogrešna — treba prvo tehnička provera (da li "
              f"je stranica uopšte živa/indeksirana), ne uređivanje teksta.")

    # 2) CTR je tačno 0 — nema nijednog klika ikad, ne samo "nizak" CTR
    elif row.ctr == 0:
        print(f"   Šta bi ga učinilo pogrešnim: CTR je tačno 0, ne samo 'nizak' — ovo je stranica koja "
              f"NIKAD nije dobila klik u 90 dana uprkos {row.impressions_90d} impresija. To je ekstremniji "
              f"slučaj od običnog 'CTR ispod proseka za poziciju' i može ukazivati na potpuno pogrešan "
              f"naslov/meta opis, ili na to da rezultat u pretrazi uopšte nije klikabilan (npr. featured "
              f"snippet krade klik). Vredi ručno otvoriti SERP pre nego što se preporuči obična izmena meta opisa.")

    # 3) Granica volumena — blizu praga od 100 impresija, signal je krhkiji
    elif row.impressions_90d < 150:
        print(f"   Šta bi ga učinilo pogrešnim: impressions_90d={row.impressions_90d} je blizu praga "
              f"od 100 koji koristimo za 'visible_enough' — na ovako malom broju impresija i CTR i "
              f"recent_share mogu biti nestabilni (par slučajnih klikova bi drastično promenilo sliku). "
              f"Signal ovde treba tretirati sa manje poverenja nego kod stranica sa hiljadama impresija.")

    # 4) Sve OK, ali proveri koncentraciju klijenata
    elif client_counts_top10.get(row.client_id, 0) >= 3:
        print(f"   Šta bi ga učinilo pogrešnim: klijent '{row.client_id}' se pojavljuje "
              f"{client_counts_top10[row.client_id]} puta u top 10 — moguće je da postoji sistemski "
              f"problem specifičan za taj nalog (npr. tehnička migracija, gubitak indeksiranosti na celom "
              f"sajtu) umesto 10 nezavisnih problema sa sadržajem. Vredi prvo proveriti da li je ceo klijent "
              f"pogođen istim uzrokom pre nego što se svaka stranica tretira posebno.")

    # 5) Standardan, "čist" slučaj — nema očiglednih crvenih zastavica
    else:
        print(f"   Šta bi ga učinilo pogrešnim: nema očiglednih crvenih zastavica u brojevima, ali i dalje "
              f"treba proveriti da li je pad sezonski (npr. tema vezana za praznik/period) pre nego što se "
              f"potroši vreme na refresh sadržaja koji možda uopšte nije problem.")

    print()

#1 | content_6dd1153d206b | action=review_content_refresh | reason=declining_visibility | score=1.000
   impressions_90d=1001, recent_share=0.00, ctr=0.0, position_tier=top_3, trend_direction=flat
   Zašto je ovde: opala vidljivost u odnosu na sopstveni 90d prosek.
   Šta bi ga učinilo pogrešnim: Reason code kaže 'opadajuća vidljivost', ali stvarni trend_direction je 'flat' — ova stranica verovatno nije opala postepeno, već je izgubila SVU vidljivost naglo negde pre više od 60 dana (možda deindeksirana, redirect, ili tehnički problem), a ne postepeni pad zbog kvaliteta sadržaja. Akcija 'refresh sadržaja' bi ovde verovatno bila pogrešna — treba prvo tehnička provera (da li je stranica uopšte živa/indeksirana), ne uređivanje teksta.

#2 | content_baaedabd1d2f | action=review_content_refresh | reason=declining_visibility | score=1.000
   impressions_90d=220, recent_share=0.00, ctr=0.0, position_tier=top_3, trend_direction=flat
   Zašto je ovde: opala vidljivost u odnosu na sopstveni 90d p

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# Leakage provera: da li je bilo šta u formuli izvedeno IZ trend_direction/trend_pct?
print("Leakage check:")
print("- baseline_action_score koristi: recent_share (iz impressions_last_30d/90d), ctr, position_tier")
print("- trend_direction/trend_pct koriste se SAMO za is_declining kolonu radi provere u tabelama, "
      "nikad kao ulaz u sam score.")
print("- content_id/client_id koriste se samo za identifikaciju/join, nikad kao feature.")

# Weak picks: pogledaj par redova gde je skor visok ali slika nije jasna
borderline = queue[(queue["impressions_90d"] >= 100) & (queue["impressions_90d"] < 150)].head(5)
print("\nGranicni slucajevi (blizu praga impressions_90d>=100) — proveri da nisu šum:")
print(borderline[output_columns])

Leakage check:
- baseline_action_score koristi: recent_share (iz impressions_last_30d/90d), ctr, position_tier
- trend_direction/trend_pct koriste se SAMO za is_declining kolonu radi provere u tabelama, nikad kao ulaz u sam score.
- content_id/client_id koriste se samo za identifikaciju/join, nikad kao feature.

Granicni slucajevi (blizu praga impressions_90d>=100) — proveri da nisu šum:
    rank            content_id          client_id  baseline_action_score  \
3      4  content_243715096698  client_f369cb89fc               1.000000   
11    12  content_0921985d4f41  client_f369cb89fc               0.995370   
18    19  content_51715e336a42  client_f74efabef1               0.991453   
36    37  content_3302e42b9b95  client_3fdba35f04               0.985714   
49    50  content_296035ee7b1f  client_19581e27de               0.980469   

                   reason_code                  action  impressions_90d  \
3         declining_visibility  review_content_refresh              143   
11

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.